# 02 - Estrutura do Repositório

## Descrição

Este notebook documenta a organização física e lógica do repositório **Engenharia_Dados_Final**. A estrutura foi projetada para separar claramente as responsabilidades: orquestração (Airflow DAGs), processamento (Spark jobs), infraestrutura (scripts), contratos (configs), documentação (MkDocs) e dados (datasets). Cada diretório tem um propósito bem definido, facilitando manutenção, onboarding e extensibilidade.

---

## Árvore de Diretórios (Nível 2)

```text
Engenharia_Dados_Final/
├── .github/
│   └── workflows/
│       ├── mkdocs.yml          # CI/CD: deploy automático da documentação
│       └── tests.yml           # CI/CD: execução automática de testes unitários
├── architecture_files/
│   ├── architecture_project.jpg
│   ├── architecture_mongoDB.jpg
│   └── *.txt                   # Descrições textuais dos diagramas
├── assets/
│   └── .gitignore              # Ignora arquivos gerados localmente
├── config/
│   ├── landing_structure.json  # Contrato da camada Landing
│   ├── bronze_structure.json   # Contrato da camada Bronze
│   ├── silver_structure.json   # Contrato da camada Silver
│   └── gold_structure.json     # Contrato da camada Gold
├── dags/
│   ├── lib/                    # Módulos Python compartilhados pelas DAGs
│   │   ├── mongodb_landing.py  # Helpers: extração MongoDB → Landing
│   │   ├── landing_bronze.py   # Helpers: conversão Landing → Bronze
│   │   ├── bronze_silver.py    # Helpers: limpeza Bronze → Silver
│   │   └── silver_gold.py      # Helpers: modelagem Silver → Gold
│   ├── mongodb_to_landing.py   # DAG 1: extração incremental do MongoDB
│   ├── landing_to_bronze.py    # DAG 2: conversão JSON → Delta Bronze
│   ├── bronze_to_silver.py     # DAG 3: limpeza e validação Bronze → Silver
│   └── silver_to_gold.py       # DAG 4: modelagem dimensional Silver → Gold
├── data/
│   └── .gitignore              # Diretório de dados gerados localmente
├── dataset/
│   ├── arquivos_csv/           # CSVs gerados automaticamente (~15MB)
│   └── scripts_py/             # Geradores de dados sintéticos
│       ├── gerar_dados.py      # Script mestre que chama todos os geradores
│       ├── carregar_mongo.py   # Carrega CSVs no MongoDB com tipos BSON
│       └── gerar_*.py          # Geradores individuais por coleção
├── docker/
│   └── .gitignore
├── docs/
│   ├── arquitetura.md          # Visão geral da arquitetura
│   ├── index.md                # Página inicial do MkDocs
│   ├── modelo_mongodb.md       # Documentação da origem MongoDB
│   ├── mongodb_atlas.md        # Guia de conexão ao Atlas compartilhado
│   ├── ambiente_airflow.md     # Guia do ambiente Airflow local
│   ├── estrutura_landing.md    # Documentação da camada Landing
│   ├── estrutura_bronze.md     # Documentação da camada Bronze
│   ├── estrutura_silver.md     # Documentação da camada Silver
│   ├── estrutura_gold.md       # Documentação da camada Gold
│   ├── dag_mongodb_landing.md  # Documentação da DAG 1
│   ├── dag_landing_bronze.md   # Documentação da DAG 2
│   ├── dag_bronze_silver.md    # Documentação da DAG 3
│   ├── dag_silver_gold.md      # Documentação da DAG 4
│   ├── referencias.md          # Referências externas e bibliografia
│   ├── stylesheets/
│   └── javascripts/
├── mongodb/
│   └── schemas/                # JSON Schemas para validação MongoDB
│       ├── clientes.schema.json
│       ├── categorias.schema.json
│       ├── fornecedores.schema.json
│       ├── produtos.schema.json
│       ├── cupons.schema.json
│       ├── pedidos.schema.json
│       ├── itens_pedido.schema.json
│       ├── pagamentos.schema.json
│       ├── entregas.schema.json
│       ├── avaliacoes.schema.json
│       └── README.md
├── notebooks/
│   └── .gitignore
├── scripts/
│   ├── lib/                    # Módulos compartilhados pelos scripts
│   │   ├── object_storage.py   # Operações S3/MinIO genéricas
│   │   └── delta_structure.py  # Operações de estrutura Delta Lake genéricas
│   ├── criar_estrutura_landing.py   # Script: cria estrutura Landing no MinIO
│   ├── criar_estrutura_bronze.py    # Script: cria estrutura Bronze no MinIO
│   ├── criar_estrutura_silver.py    # Script: cria estrutura Silver no MinIO
│   └── criar_estrutura_gold.py      # Script: cria estrutura Gold no MinIO
├── spark_jobs/
│   ├── __init__.py
│   ├── landing_to_bronze.py    # Job PySpark: Landing → Bronze
│   ├── bronze_to_silver.py     # Job PySpark: Bronze → Silver
│   └── silver_to_gold.py       # Job PySpark: Silver → Gold
├── tests/
│   ├── test_mongodb_landing.py       # Testes: helpers MongoDB → Landing
│   ├── test_landing_bronze.py        # Testes: helpers Landing → Bronze
│   ├── test_bronze_silver.py         # Testes: helpers Bronze → Silver
│   ├── test_silver_gold.py           # Testes: helpers Silver → Gold
│   ├── test_scd2_gold_spark.py       # Testes: integração SCD Tipo 2 (PySpark)
│   ├── test_airflow_environment.py   # Testes: configuração do ambiente Airflow
│   ├── test_landing_structure.py     # Testes: estrutura Landing (FakeS3Client)
│   ├── test_bronze_structure.py      # Testes: estrutura Bronze (FakeS3Client)
│   ├── test_silver_structure.py      # Testes: estrutura Silver (FakeS3Client)
│   └── test_gold_structure.py        # Testes: estrutura Gold (FakeS3Client)
├── .dockerignore
├── .env.example                # Template de variáveis de ambiente
├── .gitignore
├── .python-version             # Versão do Python (3.11+)
├── Dockerfile.airflow          # Imagem customizada do Airflow
├── docker-compose.yml          # Stack completo (MongoDB, MinIO, Airflow, Postgres)
├── LICENSE                     # Licença MIT
├── mkdocs.yml                  # Configuração do site de documentação
├── pyproject.toml              # Dependências Python (grupos por contexto)
└── README.md                   # Documentação principal do projeto
```

---

## Responsabilidades por Diretório

### `dags/` — Orquestração (Airflow DAGs)

**Responsabilidade**: Definir os grafos de execução (DAGs) que orquestram o pipeline.

| Arquivo | Função |
|---------|--------|
| `mongodb_to_landing.py` | DAG de extração incremental do MongoDB para Landing (JSON) |
| `landing_to_bronze.py` | DAG de conversão de JSON para Delta Lake na Bronze |
| `bronze_to_silver.py` | DAG de limpeza, deduplicação e validação Bronze → Silver |
| `silver_to_gold.py` | DAG de modelagem dimensional com SCD Tipo 2 na Gold |
| `lib/mongodb_landing.py` | Helpers puros: checkpoints, filtros incrementais, manifestos |
| `lib/landing_bronze.py` | Helpers puros: URIs, manifestos, configuração Spark |
| `lib/bronze_silver.py` | Helpers puros: regras de entidade, validação, qualidade |
| `lib/silver_gold.py` | Helpers puros: modelos Gold, SCD Tipo 2, manifestos |

**Princípio**: As DAGs contêm apenas a lógica de orquestração (tasks, dependências, agendamento). Toda a lógica de negócio está nos módulos `lib/`, permitindo testes unitários independentes do Airflow.

> **Referência**: `dags/mongodb_to_landing.py`, `dags/lib/mongodb_landing.py`

### `spark_jobs/` — Processamento (PySpark Jobs)

**Responsabilidade**: Implementar as transformações de dados entre camadas usando PySpark e Delta Lake.

| Arquivo | Função |
|---------|--------|
| `landing_to_bronze.py` | Lê JSON estendido da Landing, converte para Delta, adiciona metadados de auditoria |
| `bronze_to_silver.py` | Lê Delta Bronze, aplica deduplicação, validação, integridade referencial, grava Delta Silver |
| `silver_to_gold.py` | Lê Delta Silver, constrói dimensões (SCD2) e fatos, grava Delta Gold |

**Princípio**: Cada job é um script Python independente, chamado via `SparkSubmitOperator` no Airflow. Recebe argumentos de CLI (bucket, database, run_id, etc.) e grava manifestos de auditoria.

> **Referência**: `spark_jobs/landing_to_bronze.py`, `spark_jobs/silver_to_gold.py`

### `scripts/` — Infraestrutura (Setup do Data Lake)

**Responsabilidade**: Criar e validar a estrutura de diretórios/prefixos no MinIO/S3 antes das DAGs executarem.

| Arquivo | Função |
|---------|--------|
| `criar_estrutura_landing.py` | Cria bucket, prefixos e manifesto da Landing |
| `criar_estrutura_bronze.py` | Cria marcadores `_READY` e manifesto da Bronze |
| `criar_estrutura_silver.py` | Cria marcadores `_READY` e manifesto da Silver |
| `criar_estrutura_gold.py` | Cria marcadores `_READY` e manifesto da Gold |
| `lib/object_storage.py` | Operações genéricas S3 (bucket, objetos, manifestos) |
| `lib/delta_structure.py` | Operações genéricas de estrutura Delta (validação, config) |

**Princípio**: Scripts idempotentes que podem ser executados múltiplas vezes sem efeitos colaterais. Usam `boto3` para operar tanto MinIO local quanto Amazon S3.

> **Referência**: `scripts/criar_estrutura_landing.py`, `scripts/lib/object_storage.py`

### `config/` — Contratos Versionados

**Responsabilidade**: Definir o schema esperado de cada camada em formato JSON, servindo como contrato entre os scripts de infraestrutura e as DAGs.

| Arquivo | Conteúdo |
|---------|----------|
| `landing_structure.json` | Bucket, database, camada, lista de coleções |
| `bronze_structure.json` | Bucket, database, camada, tabelas, colunas de partição (`ingestion_date`) |
| `silver_structure.json` | Bucket, database, camada, tabelas, sem partição |
| `gold_structure.json` | Bucket, database, camada, dimensões e fatos, sem partição |

**Princípio**: JSON é human-readable, versionável pelo Git e consumido tanto por scripts Python quanto por testes.

> **Referência**: `config/landing_structure.json`, `config/bronze_structure.json`

### `dataset/` — Dados Sintéticos

**Responsabilidade**: Gerar e carregar os dados de exemplo no MongoDB.

| Arquivo | Função |
|---------|--------|
| `scripts_py/gerar_dados.py` | Script mestre: gera todos os CSVs chamando geradores individuais |
| `scripts_py/carregar_mongo.py` | Lê CSVs, converte tipos BSON, cria coleções com `$jsonSchema`, insere em lotes |
| `scripts_py/gerar_clientes.py` | Gera 15.000 clientes sintéticos com Faker |
| `scripts_py/gerar_pedidos.py` | Gera 15.000 pedidos com status ponderado e cupom opcional |
| `scripts_py/gerar_*.py` | Geradores para cada uma das 10 coleções |
| `arquivos_csv/` | CSVs gerados (~15MB total, não versionados) |

**Princípio**: Dados determinísticos (sementes fixas) garantem reprodutibilidade. O script `carregar_mongo.py` é idempotente (recria coleções a cada execução).

> **Referência**: `dataset/scripts_py/carregar_mongo.py`, `dataset/scripts_py/gerar_dados.py`

### `mongodb/schemas/` — Schema Validation

**Responsabilidade**: Definir os validadores `$jsonSchema` do MongoDB para cada coleção, garantindo tipos e constraints na origem.

| Arquivo | Função |
|---------|--------|
| `clientes.schema.json` | Schema de validação da coleção clientes (tipos, required, enum) |
| `pedidos.schema.json` | Schema de validação da coleção pedidos (status, id_cupom opcional) |
| `*.schema.json` | Schemas para as demais 8 coleções |
| `README.md` | Documentação dos schemas |

**Princípio**: Os schemas são aplicados automaticamente por `carregar_mongo.py` na criação das coleções, podem ser desativados via `--no-validator`.

> **Referência**: `mongodb/schemas/clientes.schema.json`, `mongodb/schemas/pedidos.schema.json`

### `tests/` — Testes Automatizados

**Responsabilidade**: Garantir a qualidade do código através de testes unitários e de integração.

| Arquivo | Escopo | Tipo |
|---------|--------|------|
| `test_mongodb_landing.py` | Helpers da DAG 1 (checkpoints, filtros, manifestos) | Unitário |
| `test_landing_bronze.py` | Helpers da DAG 2 (URIs, Spark conf, manifestos) | Unitário |
| `test_bronze_silver.py` | Helpers da DAG 3 (regras de entidade, manifestos) | Unitário |
| `test_silver_gold.py` | Helpers da DAG 4 (modelos Gold, SCD2, manifestos) | Unitário |
| `test_scd2_gold_spark.py` | Integração SCD Tipo 2 com PySpark + Delta | Integração |
| `test_airflow_environment.py` | Validação do docker-compose e pyproject.toml | Unitário |
| `test_*_structure.py` | Testes de estrutura dos scripts de infra (FakeS3Client) | Unitário |

**Princípio**: Testes unitários usam `FakeS3Client` (mock de S3) para não depender de infraestrutura. Testes de integração com PySpark são pulados automaticamente quando Spark não está instalado.

> **Referência**: `tests/test_mongodb_landing.py`, `tests/test_scd2_gold_spark.py`

### `docs/` — Documentação MkDocs

**Responsabilidade**: Fonte da documentação estática publicada via GitHub Pages.

| Arquivo | Tema |
|---------|------|
| `arquitetura.md` | Visão geral, decisões de design, princípios |
| `modelo_mongodb.md` | Modelo de dados da origem, dicionário de dados, schemas |
| `ambiente_airflow.md` | Setup, conexões, comandos úteis do Airflow |
| `estrutura_*.md` | Documentação específica de cada camada (Landing, Bronze, Silver, Gold) |
| `dag_*.md` | Documentação específica de cada DAG |
| `referencias.md` | Links externos, bibliografia, recursos |

**Princípio**: Markdown puro com extensões MkDocs (admonitions, snippets, mermaid, tabs). Publicado automaticamente via GitHub Actions.

> **Referência**: `docs/arquitetura.md`, `docs/modelo_mongodb.md`, `mkdocs.yml`

---

## Diagrama de Dependências dos Módulos

```mermaid
graph TD
    subgraph Configuração\nContratos
        CFG1[config/landing_structure.json]
        CFG2[config/bronze_structure.json]
        CFG3[config/silver_structure.json]
        CFG4[config/gold_structure.json]
    end

    subgraph Infraestrutura\nScripts
        INF1[scripts/criar_estrutura_landing.py]
        INF2[scripts/criar_estrutura_bronze.py]
        INF3[scripts/criar_estrutura_silver.py]
        INF4[scripts/criar_estrutura_gold.py]
        INF_LIB[scripts/lib/\nobject_storage.py\ndelta_structure.py]
    end

    subgraph Orquestração\nDAGs
        DAG1[dags/mongodb_to_landing.py]
        DAG2[dags/landing_to_bronze.py]
        DAG3[dags/bronze_to_silver.py]
        DAG4[dags/silver_to_gold.py]
        DAG_LIB[dags/lib/\nmongodb_landing.py\nlanding_bronze.py\nbronze_silver.py\nsilver_gold.py]
    end

    subgraph Processamento\nSpark Jobs
        SPK1[spark_jobs/landing_to_bronze.py]
        SPK2[spark_jobs/bronze_to_silver.py]
        SPK3[spark_jobs/silver_to_gold.py]
    end

    subgraph Testes
        T1[tests/test_mongodb_landing.py]
        T2[tests/test_landing_bronze.py]
        T3[tests/test_bronze_silver.py]
        T4[tests/test_silver_gold.py]
        T5[tests/test_scd2_gold_spark.py]
        T6[tests/test_*_structure.py]
    end

    CFG1 --> INF1
    CFG2 --> INF2
    CFG3 --> INF3
    CFG4 --> INF4

    INF_LIB --> INF1
    INF_LIB --> INF2
    INF_LIB --> INF3
    INF_LIB --> INF4

    DAG_LIB --> DAG1
    DAG_LIB --> DAG2
    DAG_LIB --> DAG3
    DAG_LIB --> DAG4

    DAG1 --> SPK1
    DAG2 --> SPK2
    DAG3 --> SPK3

    DAG_LIB --> T1
    DAG_LIB --> T2
    DAG_LIB --> T3
    DAG_LIB --> T4
    SPK3 --> T5
    INF_LIB --> T6

    style CFG1 fill:#e2e8f0,stroke:#94a3b8,color:#1e293b
    style INF_LIB fill:#bfdbfe,stroke:#3b82f6,color:#1e3a8a
    style DAG_LIB fill:#bfdbfe,stroke:#3b82f6,color:#1e3a8a
    style SPK1 fill:#fdba74,stroke:#b45309,color:#7c2d12
    style SPK2 fill:#cbd5e1,stroke:#64748b,color:#1e293b
    style SPK3 fill:#fde047,stroke:#a16207,color:#713f12
```

---

## Artefatos Versionados vs. Gerados

### Versionados (Git)

| Artefato | Localização | Por que versionar? |
|----------|-------------|-------------------|
| Código-fonte (DAGs, jobs, scripts, libs) | `dags/`, `spark_jobs/`, `scripts/`, `tests/` | Lógica de negócio, testes, infraestrutura |
| Configurações (JSON) | `config/` | Contratos versionáveis entre camadas |
| Schemas MongoDB | `mongodb/schemas/` | Definição da origem, validação |
| Geradores de dados | `dataset/scripts_py/` | Lógica de geração determinística |
| Documentação (MkDocs) | `docs/` | Fonte do site de documentação |
| Docker e Compose | `docker-compose.yml`, `Dockerfile.airflow` | Definição do ambiente |
| CI/CD | `.github/workflows/` | Automação de testes e deploy |
| Documentação (notebooks) | `notebooks/` | Este conjunto de documentação |

### Não Versionados (Git Ignore)

| Artefato | Localização | Por que ignorar? |
|----------|-------------|-----------------|
| Dados CSV gerados | `dataset/arquivos_csv/` | Reproduzíveis (~15MB), gerados por script |
| Dados de execução | `data/` | Dados gerados localmente durante execução |
| Logs do Airflow | `airflow/logs/` | Logs gerados em runtime |
| Site do MkDocs | `site/` | Gerado por `mkdocs build`, publicado via gh-deploy |
| Variáveis de ambiente | `.env` | Contém credenciais e configurações sensíveis |
| Cache Python | `__pycache__/`, `.pyc` | Artefatos de compilação |
